In [2]:
"""
Pi-Rating Walk-Forward Hyperparameter Tuning
=============================================
Input  : C:/Users/semwi/FPL-Core-Insights/data/match_data.csv
Output : C:/Users/semwi/FPL-Core-Insights/data/Kalman data/optuna_pi_results.csv
         C:/Users/semwi/FPL-Core-Insights/data/Kalman data/optuna_pi_best_params.json
         C:/Users/semwi/FPL-Core-Insights/data/Kalman data/pi_ratings_tuned.csv

Doel
----
De Pi-ratings worden gebruikt als features in de downstream Kalman filter.
De tuning optimaliseert uitsluitend de kwaliteit van pi_expected_gd_pre als
voorspeller van het werkelijke doelpuntenverschil (home_goals - away_goals).

Objective : Mean Absolute Error (MAE) op doelpuntenverschil,
            geëvalueerd via expanding-window walk-forward validatie.

Hyperparameter vector θ_π = (λ_main, λ_cross, home_adv, squash_scale)

Referenties
-----------
Constantinou, A.C. & Fenton, N.E. (2013). Determining the level of ability
  of football teams by dynamic ratings based on the relative discrepancies
  in scores between adversaries. Journal of Quantitative Analysis in Sports.
"""

import os, math, json, warnings
import pandas as pd
import numpy as np
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# 0.  Paden
# ─────────────────────────────────────────────────────────────────────────────
BASE_DIR   = r"C:\Users\semwi\FPL-Core-Insights\data"
OUTPUT_DIR = os.path.join(BASE_DIR, "Kalman data")
MATCH_PATH = os.path.join(BASE_DIR, "match_data.csv")
os.makedirs(OUTPUT_DIR, exist_ok=True)

UD_TEAM_MAP = {
    "Tottenham":     "Tottenham Hotspur",
    "Newcastle":     "Newcastle United",
    "Wolves":        "Wolverhampton Wanderers",
    "Wolverhampton": "Wolverhampton Wanderers",
    "Brighton":      "Brighton & Hove Albion",
    "West Ham":      "West Ham United",
    "Leicester":     "Leicester City",
    "Ipswich":       "Ipswich Town",
    "Luton":         "Luton Town",
    "Norwich":       "Norwich City",
    "Leeds":         "Leeds United",
}

# ─────────────────────────────────────────────────────────────────────────────
# 1.  Pi-Rating systeem
# ─────────────────────────────────────────────────────────────────────────────

class PiRatingSystem:
    """
    Pi-ratingsysteem (Constantinou & Fenton, 2013).

    Elk team heeft twee ratings:
      - home_rating : prestatiesterkte als thuisspelend team
      - away_rating : prestatiesterkte als uitspelend team

    De cross-update zorgt dat thuis- en uitrating naar elkaar convergeren,
    wat een impliciete regularisatie vormt tegen context-overfitting.
    """

    def __init__(self, lambda_main: float, lambda_cross: float,
                 home_adv: float, squash_scale: float):
        self.lambda_main  = lambda_main
        self.lambda_cross = lambda_cross
        self.home_adv     = home_adv
        self.squash_scale = squash_scale
        self.home_ratings = {}
        self.away_ratings = {}

    def _get(self, team):
        return (self.home_ratings.get(team, 0.0),
                self.away_ratings.get(team, 0.0))

    def _squash(self, error: float) -> float:
        """tanh-squashing om grote doelpuntenmarges af te vlakken."""
        return math.tanh(error / self.squash_scale) * self.squash_scale

    def expected_goal_diff(self, home_team: str, away_team: str) -> float:
        """Pre-match verwacht doelpuntenverschil (thuis minus uit)."""
        h_home, _ = self._get(home_team)
        _, a_away = self._get(away_team)
        return h_home - a_away + self.home_adv

    def update(self, home_team: str, away_team: str, actual_gd: float):
        """Update ratings na een gespeelde wedstrijd."""
        h_home, h_away = self._get(home_team)
        a_home, a_away = self._get(away_team)

        expected   = h_home - a_away + self.home_adv
        se         = self._squash(actual_gd - expected)

        new_h_home = h_home + self.lambda_main * se
        new_a_away = a_away - self.lambda_main * se
        new_h_away = h_away + self.lambda_cross * (new_h_home - h_away)
        new_a_home = a_home + self.lambda_cross * (new_a_away - a_away)

        self.home_ratings[home_team] = new_h_home
        self.away_ratings[home_team] = new_h_away
        self.home_ratings[away_team] = new_a_home
        self.away_ratings[away_team] = new_a_away


# ─────────────────────────────────────────────────────────────────────────────
# 2.  Data laden
# ─────────────────────────────────────────────────────────────────────────────

def load_data(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, low_memory=False)
    df["home_team"] = df["home_team"].replace(UD_TEAM_MAP)
    df["away_team"] = df["away_team"].replace(UD_TEAM_MAP)
    df["date"]      = pd.to_datetime(df["timestamp"], format="mixed", errors="coerce")
    df = df.sort_values("date").reset_index(drop=True)
    df = df.dropna(subset=["home_goals", "away_goals"]).copy()
    df["home_goals"] = df["home_goals"].astype(int)
    df["away_goals"] = df["away_goals"].astype(int)
    df["season"]     = df["season"].str.strip()
    return df


# ─────────────────────────────────────────────────────────────────────────────
# 3.  Walk-forward validatie  —  objective: MAE op doelpuntenverschil
# ─────────────────────────────────────────────────────────────────────────────

def walk_forward_pi(
    df: pd.DataFrame,
    lambda_main:  float,
    lambda_cross: float,
    home_adv:     float,
    squash_scale: float,
    min_train_seasons: int = 2,
) -> dict:
    """
    Expanding-window walk-forward validatie voor het Pi-ratingsysteem.

    Per fold:
      1. Warm-up  : bouw pi-ratings op alle voorgaande seizoenen
      2. Valideer : bereken pre-match pi_expected_gd_pre per wedstrijd
                    → MAE vs werkelijk doelpuntenverschil
                    → update ratings na elke wedstrijd (strikt causaal)
    """
    seasons = sorted(df["season"].unique())
    fold_results = []

    for val_idx in range(min_train_seasons, len(seasons)):
        train_df = df[df["season"].isin(seasons[:val_idx])].copy()
        val_df   = df[df["season"] == seasons[val_idx]].copy()

        if len(val_df) == 0:
            continue

        # Warm-up
        pi = PiRatingSystem(lambda_main, lambda_cross, home_adv, squash_scale)
        for _, row in train_df.iterrows():
            pi.update(row["home_team"], row["away_team"],
                      row["home_goals"] - row["away_goals"])

        # Out-of-sample MAE
        mae_fold = []
        for _, row in val_df.iterrows():
            h, a      = row["home_team"], row["away_team"]
            exp_gd    = pi.expected_goal_diff(h, a)
            actual_gd = row["home_goals"] - row["away_goals"]
            mae_fold.append(abs(exp_gd - actual_gd))
            pi.update(h, a, actual_gd)

        fold_results.append({
            "fold"    : seasons[val_idx],
            "mean_mae": np.mean(mae_fold),
            "n"       : len(mae_fold),
        })

    fold_df  = pd.DataFrame(fold_results)
    mean_mae = fold_df["mean_mae"].mean() if len(fold_df) > 0 else 999.0

    return {
        "mean_mae"   : mean_mae,
        "std_mae"    : fold_df["mean_mae"].std() if len(fold_df) > 0 else 0.0,
        "n_folds"    : len(fold_df),
        "fold_detail": fold_df,
    }


# ─────────────────────────────────────────────────────────────────────────────
# 4.  Optuna tuning
# ─────────────────────────────────────────────────────────────────────────────

def run_optuna_tuning(df: pd.DataFrame, n_trials: int = 200) -> tuple:
    """Bayesiaanse hyperparameter optimalisatie via Optuna TPE sampler."""
    call_log = []

    def objective(trial):
        lambda_main  = trial.suggest_float("lambda_main",  0.001, 0.50)
        lambda_cross = trial.suggest_float("lambda_cross", 0.10,  0.99)
        home_adv     = trial.suggest_float("home_adv",     0.00,  0.60)
        squash_scale = trial.suggest_float("squash_scale", 0.5,   8.0)

        result = walk_forward_pi(df, lambda_main, lambda_cross,
                                 home_adv, squash_scale)

        call_log.append({
            "trial"       : trial.number,
            "lambda_main" : lambda_main,
            "lambda_cross": lambda_cross,
            "home_adv"    : home_adv,
            "squash_scale": squash_scale,
            "mean_mae"    : result["mean_mae"],
            "std_mae"     : result["std_mae"],
        })

        return result["mean_mae"]

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=42),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    results_df = pd.DataFrame(call_log).sort_values("mean_mae")
    return results_df, study.best_params


# ─────────────────────────────────────────────────────────────────────────────
# 5.  Pi-ratings herbouwen met optimale parameters
# ─────────────────────────────────────────────────────────────────────────────

def build_pi_ratings(df: pd.DataFrame, lambda_main: float,
                     lambda_cross: float, home_adv: float,
                     squash_scale: float) -> pd.DataFrame:
    """
    Bouw volledige pi-rating tijdreeks met optimale parameters.
    Alle ratings zijn pre-match (strikt causaal, geen look-ahead).
    """
    pi = PiRatingSystem(lambda_main, lambda_cross, home_adv, squash_scale)
    records = []

    for _, row in df.iterrows():
        h, a = row["home_team"], row["away_team"]

        h_home_pre, _ = pi._get(h)
        _, a_away_pre = pi._get(a)
        exp_gd_pre    = pi.expected_goal_diff(h, a)

        records.append({
            "match_id"          : row.get("match_id", None),
            "date"              : row["date"].date() if hasattr(row["date"], "date") else row["date"],
            "season"            : row["season"],
            "home_team"         : h,
            "away_team"         : a,
            "home_goals"        : row["home_goals"],
            "away_goals"        : row["away_goals"],
            "home_pi_home_pre"  : round(h_home_pre, 5),
            "away_pi_away_pre"  : round(a_away_pre, 5),
            "pi_expected_gd_pre": round(exp_gd_pre, 5),
        })

        pi.update(h, a, row["home_goals"] - row["away_goals"])

    return pd.DataFrame(records)


# ─────────────────────────────────────────────────────────────────────────────
# 6.  Main
# ─────────────────────────────────────────────────────────────────────────────

def main():
    print("=" * 60)
    print("  Pi-Rating Walk-Forward Hyperparameter Tuning (Optuna)")
    print("=" * 60)

    print("\nData laden...")
    df = load_data(MATCH_PATH)
    print(f"  {len(df)} gespeelde wedstrijden")
    print(f"  Seizoenen: {sorted(df['season'].unique())}")

    print(f"\nOptuna hyperparameter tuning (200 trials)...")
    print("  Zoekruimte:")
    print("    λ_main       : [0.001, 0.50]")
    print("    λ_cross      : [0.10,  0.99]")
    print("    home_adv     : [0.00,  0.60]")
    print("    squash_scale : [0.5,   8.0]")
    print("  Objective     : MAE op doelpuntenverschil\n")

    results_df, _ = run_optuna_tuning(df, n_trials=200)
    best = results_df.iloc[0]

    print(f"\n{'─'*50}")
    print(f"Optimale hyperparameters (θ_π*):")
    print(f"  λ_main       = {best['lambda_main']:.5f}")
    print(f"  λ_cross      = {best['lambda_cross']:.5f}")
    print(f"  home_adv     = {best['home_adv']:.5f}")
    print(f"  squash_scale = {best['squash_scale']:.5f}")
    print(f"\n  Mean MAE = {best['mean_mae']:.4f}")
    print(f"  Std  MAE = {best['std_mae']:.4f}")
    print(f"{'─'*50}")

    # Vergelijk met huidige parameters
    current = walk_forward_pi(df, 0.05, 0.70, 0.20, 3.0)
    print(f"\nVergelijking met huidige parameters (λ=0.05, cross=0.70, adv=0.20, squash=3.0):")
    print(f"  Huidig   MAE = {current['mean_mae']:.4f}")
    print(f"  Optimaal MAE = {best['mean_mae']:.4f}")
    print(f"  Verbetering  = {(current['mean_mae'] - best['mean_mae'])/current['mean_mae']*100:.2f}%")

    # Per-fold detail
    print(f"\nPer-fold MAE (optimale θ_π*):")
    final = walk_forward_pi(df, best["lambda_main"], best["lambda_cross"],
                            best["home_adv"], best["squash_scale"])
    print(final["fold_detail"].to_string(index=False))

    # Opslaan
    results_path = os.path.join(OUTPUT_DIR, "optuna_pi_results.csv")
    params_path  = os.path.join(OUTPUT_DIR, "optuna_pi_best_params.json")

    results_df.to_csv(results_path, index=False)

    best_out = {
        "lambda_main" : float(best["lambda_main"]),
        "lambda_cross": float(best["lambda_cross"]),
        "home_adv"    : float(best["home_adv"]),
        "squash_scale": float(best["squash_scale"]),
        "mean_mae"    : float(best["mean_mae"]),
        "std_mae"     : float(best["std_mae"]),
    }
    with open(params_path, "w") as f:
        json.dump(best_out, f, indent=2)

    print(f"\nOpgeslagen:")
    print(f"  {results_path}")
    print(f"  {params_path}")

    # Pi-ratings herbouwen
    print(f"\nPi-ratings herbouwen met optimale parameters...")
    pi_df   = build_pi_ratings(df, best["lambda_main"], best["lambda_cross"],
                                best["home_adv"], best["squash_scale"])
    pi_path = os.path.join(OUTPUT_DIR, "pi_ratings_tuned.csv")
    pi_df.to_csv(pi_path, index=False)
    print(f"  Opgeslagen: {pi_path}  ({len(pi_df)} rijen)")
    print(f"\nKlaar! Gebruik optuna_pi_best_params.json in de Kalman updater.")
    return best_out


if __name__ == "__main__":
    best = main()

  Pi-Rating Walk-Forward Hyperparameter Tuning (Optuna)

Data laden...
  3527 gespeelde wedstrijden
  Seizoenen: ['2016-2017', '2017-2018', '2019-2020', '2020-2021', '2021-2022', '2022-2023', '2024-2025', '2025-2026', '23/24']

Optuna hyperparameter tuning (200 trials)...
  Zoekruimte:
    λ_main       : [0.001, 0.50]
    λ_cross      : [0.10,  0.99]
    home_adv     : [0.00,  0.60]
    squash_scale : [0.5,   8.0]
  Objective     : MAE op doelpuntenverschil



  0%|          | 0/200 [00:00<?, ?it/s]


──────────────────────────────────────────────────
Optimale hyperparameters (θ_π*):
  λ_main       = 0.03530
  λ_cross      = 0.98898
  home_adv     = 0.20285
  squash_scale = 1.56795

  Mean MAE = 1.3226
  Std  MAE = 0.0587
──────────────────────────────────────────────────

Vergelijking met huidige parameters (λ=0.05, cross=0.70, adv=0.20, squash=3.0):
  Huidig   MAE = 1.3271
  Optimaal MAE = 1.3226
  Verbetering  = 0.34%

Per-fold MAE (optimale θ_π*):
     fold  mean_mae   n
2019-2020  1.260953 559
2020-2021  1.362983 380
2021-2022  1.334471 380
2022-2023  1.350970 380
2024-2025  1.348632 380
2025-2026  1.220817 308
    23/24  1.379682 380

Opgeslagen:
  C:\Users\semwi\FPL-Core-Insights\data\Kalman data\optuna_pi_results.csv
  C:\Users\semwi\FPL-Core-Insights\data\Kalman data\optuna_pi_best_params.json

Pi-ratings herbouwen met optimale parameters...
  Opgeslagen: C:\Users\semwi\FPL-Core-Insights\data\Kalman data\pi_ratings_tuned.csv  (3527 rijen)

Klaar! Gebruik optuna_pi_best_par